# 00_pilot — 학습 곡선 pilot, M0 선택, stream 봉인 (RF-06)

[RF-06 계획서](../../../docs/ml/VERSION_POLICY_EFFICIENCY_EXPERIMENT.md) 3절. 이 노트북이 끝나면 실험의 모든 데이터가 고정된다. 이후 노트북은 아무것도 고르지 않는다.

1. **시험(smoke):** 전체 경로(pilot → 봉인 → P1·P2 round → 비교)를 작은 크기, 2 step으로 `/content`에서 한 번 돌린다. Drive에 쓰지 않는다. 약 10분.
2. **pilot:** pool에서 1k·2k·4k·8k·16k행(원천 비율 유지, 중복 그룹 통째로, 작은 표본은 큰 표본에 포함)으로 ModernBERT-base를 3 epoch 학습하고 dev source-macro-F1을 잰다. 크기마다 끝나면 Drive `pilot/n<크기>/`에 저장하고, 다시 실행하면 끝난 크기는 건너뛴다. **약 2.5~3시간.**
3. **봉인:** 규칙대로 n0(M0 크기)와 stream 크기를 정한다. M0로 probe window를 채점해 9 round의 재학습 batch(틀린 문항 전부 + reserve에서 라벨 비율을 맞춘 채움)를 만들고 `plan/`에 SHA와 함께 쓴다. **약 15분.** 규칙이 실패하면(stream이 .03 미만으로 오름) 멈추고 설계를 다시 정한다.

**런타임: 2026.07(Python 3.12), T4 GPU.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/version-policy/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'version-policy-source.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/version-policy-source')
shutil.rmtree(REPO, ignore_errors=True)
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'

subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ml/notebooks/version_policy/requirements-colab.txt')], check=True)

설치는 새 프로세스에서 바로 쓰이므로 세션을 다시 시작하지 않아도 된다. 연결이 끊긴 뒤 다시 열면 위 두 셀부터 다시 실행한다.

In [ ]:
CODE = REPO / 'ml/notebooks/version_policy'
DATA = RUN / 'data'
ENV = dict(os.environ, PYTHONPATH=str(REPO / 'ml/src'), PYTHONUNBUFFERED='1', TRANSFORMERS_VERBOSITY='error')

def run_logged(script, args):
    """Run a script in a fresh process (it sees the packages installed above) and stream its log here."""
    proc = subprocess.Popen([sys.executable, str(CODE / script), *map(str, args)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=ENV, cwd=str(CODE))
    for line in proc.stdout:
        if 'Warning' not in line and 'warn(' not in line:
            print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

assert (DATA / 'manifest.json').exists(), 'Drive에 data/(로컬 build 결과)를 먼저 올리세요. README 참고'
print(json.loads((DATA / 'manifest.json').read_text())['rows'])

import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요'
print(torch.cuda.get_device_name(0), torch.__version__)

## 1. 시험 (smoke, 약 10분)

`/content/rf06-smoke-run`에만 쓴다. 숫자는 의미가 없고 경로가 끝까지 도는지만 본다.

In [ ]:
SMOKE = Path('/content/rf06-smoke-run')
shutil.rmtree(SMOKE, ignore_errors=True)
run_logged('plan.py', ['pilot', '--repo', REPO, '--data-dir', DATA, '--run-dir', SMOKE, '--smoke', '--device', 'cuda'])
run_logged('plan.py', ['seal', '--repo', REPO, '--data-dir', DATA, '--run-dir', SMOKE, '--smoke', '--n0', 200, '--device', 'cuda'])
for arm in ('p1', 'p2', 'p2_eq'):
    run_logged('rounds.py', ['--repo', REPO, '--data-dir', DATA, '--run-dir', SMOKE, '--arm', arm, '--smoke', '--device', 'cuda'])
run_logged('compare.py', ['--run-dir', SMOKE, '--data-dir', DATA, '--output-dir', SMOKE / 'compare'])
print((SMOKE / 'compare' / 'REPORT-draft.md').read_text()[:1500])

## 2. pilot (약 2.5~3시간)

크기마다 끝나면 Drive에 저장된다. 끊기면 이 셀만 다시 실행하면 끝난 크기는 건너뛴다.

In [ ]:
run_logged('plan.py', ['pilot', '--repo', REPO, '--data-dir', DATA, '--run-dir', RUN, '--device', 'cuda'])
for folder in sorted((RUN / 'pilot').glob('n*'), key=lambda p: int(p.name[1:])):
    r = json.loads((folder / 'receipt.json').read_text())
    print(f"{folder.name:>7}: rows {r['actual_rows']:>6}, steps {r['optimizer_steps']:>5}, fit {r['fit_seconds']/60:5.1f}분, "
          f"dev source-macro-F1 {r['dev']['source_macro_f1']:.4f}")

## 3. 봉인 (약 15분)

`plan/`이 이미 있으면 멈춘다. 결과를 보고 다시 봉인하지 않는다.

In [ ]:
run_logged('plan.py', ['seal', '--repo', REPO, '--data-dir', DATA, '--run-dir', RUN, '--device', 'cuda'])
plan = json.loads((RUN / 'plan' / 'plan.json').read_text())
print(json.dumps({k: plan[k] for k in ('pilot_curve', 'always_easy', 'selection', 'window_rows', 'rows')}, indent=1))
for r in plan['rounds']:
    print(f"round {r['round']} ({plan['versions'][str(r['round'])]}): window {r['window_rows']}, wrong {r['wrong']}, "
          f"fill {r['fill']}, batch {r['batch_rows']} {r['batch_by_label']}")

## 4. (선택) Drive 용량 정리

M0는 `plan/m0-model/`에 복사됐다. pilot 모델 가중치(크기마다 약 600MB)는 더 쓰지 않는다. 점수·receipt는 남기고 가중치만 지우려면 `DELETE = True`로 바꿔 실행한다. 지운 파일은 Drive 휴지통으로 간다.

In [ ]:
DELETE = False
assert (RUN / 'plan' / '_SUCCESS').exists()
for model in sorted((RUN / 'pilot').glob('n*/model')):
    print('삭제' if DELETE else '삭제 예정(DELETE=False)', model)
    if DELETE:
        shutil.rmtree(model)